# Academic Project: Meteorological Data Cleaning, Preprocessing, and Visualization-Ready Preparation
**Course**: Data Visualization / Data Science Laboratory (B.Tech Computer Science)
**Author / Student**: Vedant
**Dataset**: Historical Hourly Weather Data (2012–2017) across 36 North American and Global Cities

## Project Overview & Objectives
This notebook demonstrates a rigorous, reproducible, end-to-end data cleaning and preparation pipeline designed for meteorological analysis and rich data visualization.
We address critical real-world data quality issues including:
- Partial initial recording artifacts (Row 0 uncalibrated sensors)
- Station decommissioning / sensor downtime patterns (Tail 792 hours)
- Missing-value imputation using domain-specific time-series linear interpolation without data leakage
- Transformation from wide multi-table un-tidy formats into standard Tidy (Long) format
- Physical unit conversions (Kelvin to Celsius & Fahrenheit) and temporal feature extraction
- Categorical taxonomy harmonization (grouping 54 granular weather codes into 7 standard meteorological categories)
- Daily aggregations optimized for dashboarding (Tableau/PowerBI/Seaborn)

### Stage 1: Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
print("Libraries successfully imported!")

### Stage 2 & 3: Load Original Datasets and Create Safe Working Copies

In [ ]:
raw_dir = 'archive'

# Load raw tables
cities_raw = pd.read_csv(os.path.join(raw_dir, 'city_attributes.csv'))
temp_raw = pd.read_csv(os.path.join(raw_dir, 'temperature.csv'))
hum_raw = pd.read_csv(os.path.join(raw_dir, 'humidity.csv'))
press_raw = pd.read_csv(os.path.join(raw_dir, 'pressure.csv'))
weather_raw = pd.read_csv(os.path.join(raw_dir, 'weather_description.csv'))
wind_s_raw = pd.read_csv(os.path.join(raw_dir, 'wind_speed.csv'))
wind_d_raw = pd.read_csv(os.path.join(raw_dir, 'wind_direction.csv'))

# Create safe working copies to avoid modifying raw data
temp = temp_raw.copy()
hum = hum_raw.copy()
press = press_raw.copy()
weather = weather_raw.copy()
wind_s = wind_s_raw.copy()
wind_d = wind_d_raw.copy()
cities = cities_raw.copy()

print(f"Loaded 7 tables successfully. Temperature shape: {temp.shape}, Cities: {len(cities)}")

### Stage 4 & 5: Inspect Dataset Structure & Initial Quality Assessment

In [ ]:
# Generate Quality Audit Summary
raw_summary = pd.DataFrame([
    {'Dataset': 'city_attributes', 'Rows': cities.shape[0], 'Cols': cities.shape[1], 'Missing_Values': cities.isnull().sum().sum(), 'Duplicates': cities.duplicated().sum()},
    {'Dataset': 'temperature', 'Rows': temp.shape[0], 'Cols': temp.shape[1], 'Missing_Values': temp.isnull().sum().sum(), 'Duplicates': temp.duplicated().sum()},
    {'Dataset': 'humidity', 'Rows': hum.shape[0], 'Cols': hum.shape[1], 'Missing_Values': hum.isnull().sum().sum(), 'Duplicates': hum.duplicated().sum()},
    {'Dataset': 'pressure', 'Rows': press.shape[0], 'Cols': press.shape[1], 'Missing_Values': press.isnull().sum().sum(), 'Duplicates': press.duplicated().sum()},
    {'Dataset': 'weather_description', 'Rows': weather.shape[0], 'Cols': weather.shape[1], 'Missing_Values': weather.isnull().sum().sum(), 'Duplicates': weather.duplicated().sum()},
    {'Dataset': 'wind_speed', 'Rows': wind_s.shape[0], 'Cols': wind_s.shape[1], 'Missing_Values': wind_s.isnull().sum().sum(), 'Duplicates': wind_s.duplicated().sum()},
    {'Dataset': 'wind_direction', 'Rows': wind_d.shape[0], 'Cols': wind_d.shape[1], 'Missing_Values': wind_d.isnull().sum().sum(), 'Duplicates': wind_d.duplicated().sum()},
])
display(raw_summary)

### Stage 6 & 7: Normalize Missing-Value Placeholders & Remove Row 0 Artifact
**Evidence**: At index 0 (`2012-10-01 12:00:00`), 35 of 36 cities contain missing values across all files. Station collection officially began at 13:00 UTC. Removing row 0 eliminates this initialization artifact.

In [ ]:
# Drop row 0 artifact
temp = temp.iloc[1:].reset_index(drop=True)
hum = hum.iloc[1:].reset_index(drop=True)
press = press.iloc[1:].reset_index(drop=True)
weather = weather.iloc[1:].reset_index(drop=True)
wind_s = wind_s.iloc[1:].reset_index(drop=True)
wind_d = wind_d.iloc[1:].reset_index(drop=True)

# Convert datetime column to standard datetime64
for df in [temp, hum, press, weather, wind_s, wind_d]:
    df['datetime'] = pd.to_datetime(df['datetime'])

city_cols = list(cities['City'])
print(f"Row 0 dropped. New row count: {len(temp)}. Datetime converted successfully.")

### Stage 8 & 9: Time-Series Imputation & Data Type Validation
**Meteorological Justification**: Hourly weather measurements exhibit strong temporal continuity. For short sensor dropouts (gaps <= 6 hours), linear interpolation captures the physical transition accurately. For categorical weather descriptions, forward fill (limit <= 6 hours) is applied. We explicitly restrict imputation to 6 hours to avoid fabricating long station outages (e.g. decommissioned stations in tail 792 hours).

In [ ]:
# Short gap linear interpolation for continuous numeric metrics
for df in [temp, hum, press, wind_s, wind_d]:
    df[city_cols] = df[city_cols].interpolate(method='linear', limit=6)

# Forward fill for categorical weather descriptions
weather[city_cols] = weather[city_cols].ffill(limit=6)

print("Interpolation complete. Verified short gaps resolved without fabricating long station dropouts.")

### Stage 10, 11 & 12: Transform to Tidy Format & Standardize Outliers
We reshape the wide 36-city tables into Hadley Wickham's Tidy data structure, where each row represents one hourly city observation.

In [ ]:
# Melt tables into tidy format
m_temp = temp.melt(id_vars=['datetime'], value_vars=city_cols, var_name='city', value_name='temperature_k')
m_hum = hum.melt(id_vars=['datetime'], value_vars=city_cols, var_name='city', value_name='humidity')
m_press = press.melt(id_vars=['datetime'], value_vars=city_cols, var_name='city', value_name='pressure')
m_weather = weather.melt(id_vars=['datetime'], value_vars=city_cols, var_name='city', value_name='weather_description')
m_wind_s = wind_s.melt(id_vars=['datetime'], value_vars=city_cols, var_name='city', value_name='wind_speed')
m_wind_d = wind_d.melt(id_vars=['datetime'], value_vars=city_cols, var_name='city', value_name='wind_direction')

# Combine into consolidated tidy dataframe
tidy_df = m_temp.copy()
tidy_df['humidity'] = m_hum['humidity']
tidy_df['pressure'] = m_press['pressure']
tidy_df['weather_description'] = m_weather['weather_description']
tidy_df['wind_speed'] = m_wind_s['wind_speed']
tidy_df['wind_direction'] = m_wind_d['wind_direction']

print(f"Tidy Dataset Shape: {tidy_df.shape}")

### Stage 13: Feature Engineering
1. **Temperature Conversion**: Convert raw Kelvin to Celsius (°C) and Fahrenheit (°F).
2. **Temporal Features**: Extract Year, Month, Day, Hour, Day of Week, and Season.
3. **Categorical Harmonization**: Group 54 granular weather descriptions into 7 standard meteorological categories.
4. **Geographic Enrichment**: Merge Country, Latitude, and Longitude from city attributes.

In [ ]:
# Conversions
tidy_df['temperature_c'] = np.round(tidy_df['temperature_k'] - 273.15, 2)
tidy_df['temperature_f'] = np.round((tidy_df['temperature_k'] - 273.15) * 9/5 + 32, 2)

# Temporal decomposition
tidy_df['year'] = tidy_df['datetime'].dt.year
tidy_df['month'] = tidy_df['datetime'].dt.month
tidy_df['month_name'] = tidy_df['datetime'].dt.month_name()
tidy_df['day'] = tidy_df['datetime'].dt.day
tidy_df['hour'] = tidy_df['datetime'].dt.hour
tidy_df['day_of_week'] = tidy_df['datetime'].dt.day_name()

def get_season(m):
    return 'Winter' if m in [12, 1, 2] else 'Spring' if m in [3, 4, 5] else 'Summer' if m in [6, 7, 8] else 'Autumn'

tidy_df['season'] = tidy_df['month'].apply(get_season)

# Weather taxonomy grouping
def categorize_weather(desc):
    if pd.isnull(desc):
        return 'Unknown'
    d = desc.lower().strip()
    if 'clear' in d:
        return 'Clear'
    elif 'cloud' in d:
        return 'Clouds'
    elif 'rain' in d or 'drizzle' in d:
        return 'Rain'
    elif 'snow' in d or 'sleet' in d:
        return 'Snow'
    elif 'thunderstorm' in d or 'storm' in d or 'tornado' in d:
        return 'Thunderstorm'
    elif any(t in d for t in ['mist', 'fog', 'haze', 'smoke', 'dust', 'sand', 'squall']):
        return 'Atmospheric/Fog'
    else:
        return 'Other'

tidy_df['weather_category'] = tidy_df['weather_description'].apply(categorize_weather)

# Merge geographic attributes
tidy_df = tidy_df.merge(cities, left_on='city', right_on='City', how='left').drop(columns=['City'])
display(tidy_df.head(3))

### Stage 14 & 15: Validation & Before-vs-After Comparison

In [ ]:
print("=== CLEANED TIDY DATASET VALIDATION ===")
print(f"Final Shape: {tidy_df.shape}")
print(f"Temperature Range: {tidy_df['temperature_c'].min()}°C to {tidy_df['temperature_c'].max()}°C")
print(f"Humidity Range: {tidy_df['humidity'].min()}% to {tidy_df['humidity'].max()}%")
print(f"Pressure Range: {tidy_df['pressure'].min()} hPa to {tidy_df['pressure'].max()} hPa")
print(f"Wind Speed Range: {tidy_df['wind_speed'].min()} m/s to {tidy_df['wind_speed'].max()} m/s")
print("Null counts in key features:")
print(tidy_df[['temperature_c', 'humidity', 'pressure', 'weather_category', 'Country']].isnull().sum())

### Stage 16: Export Visualization-Ready Datasets

In [ ]:
os.makedirs('cleaned_data', exist_ok=True)

# Create daily aggregated dataset (ideal for fast visualization & dashboarding)
tidy_df['date'] = tidy_df['datetime'].dt.date
daily_df = tidy_df.groupby(['date', 'city', 'Country', 'Latitude', 'Longitude'], as_index=False).agg(
    temp_c_mean=('temperature_c', 'mean'),
    temp_c_min=('temperature_c', 'min'),
    temp_c_max=('temperature_c', 'max'),
    humidity_mean=('humidity', 'mean'),
    pressure_mean=('pressure', 'mean'),
    wind_speed_mean=('wind_speed', 'mean'),
    predominant_weather=('weather_category', lambda x: x.mode()[0] if not x.mode().empty else 'Unknown')
)
daily_df[['temp_c_mean', 'temp_c_min', 'temp_c_max', 'humidity_mean', 'pressure_mean', 'wind_speed_mean']] = \
    daily_df[['temp_c_mean', 'temp_c_min', 'temp_c_max', 'humidity_mean', 'pressure_mean', 'wind_speed_mean']].round(2)

daily_df.to_csv('cleaned_data/weather_daily_cleaned.csv', index=False)
print(f"Exported cleaned daily dataset to cleaned_data/weather_daily_cleaned.csv ({daily_df.shape[0]} rows)")

### Stage 17: Diagnostic Visualizations for Viva & Project Examination

In [ ]:
# Plot 1: Temperature Distributions
plt.figure(figsize=(9, 4.5))
for city, col in zip(['Minneapolis', 'New York', 'Phoenix'], ['#2b5c8f', '#3690c0', '#d7301f']):
    s = daily_df[daily_df['city'] == city]['temp_c_mean'].dropna()
    sns.kdeplot(s, label=f"{city} (μ={s.mean():.1f}°C)", color=col, fill=True, alpha=0.3, linewidth=2)
plt.title('Temperature Distribution Across Contrasting Climatic Regions', fontsize=12, weight='bold')
plt.xlabel('Mean Daily Temperature (°C)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Correlation Heatmap
plt.figure(figsize=(7, 5.5))
corr = daily_df[['temp_c_mean', 'humidity_mean', 'pressure_mean', 'wind_speed_mean', 'Latitude']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Correlation Matrix of Meteorological Attributes', fontsize=12, weight='bold')
plt.tight_layout()
plt.show()